Developed a churn model and performed feature selection to ensure the business focused on high-impact, actionable variables rather than noisy correlations.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

load_dotenv()
engine = create_engine(os.getenv("SUPABASE_DB_URL"))

def load_cleaned():
    return pd.read_sql("SELECT * FROM telco_customer.stg_churn_cleaned", engine)

# 1. Load the data cleaned in the EDA
df = load_cleaned()

# 2. Separate the "Answer" (y) from the "Clues" (X)
# We drop customer_id (useless) and gender/phone_service (from our EDA findings)
X = df.drop(columns=['churn', 'customer_id', 'gender', 'phone_service'])
y = df['churn']

# 3. Randomly extract 20% for the "Final Exam"
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"We are training on {X_train.shape[0]} customers.")
print(f"We are testing on {X_test.shape[0]} customers.")

We are training on 5625 customers.
We are testing on 1407 customers.


In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Identify which columns need which treatment
# We'll use the columns currently in your X_train
numeric_features = ['tenure', 'monthly_charges', 'total_charges']
categorical_features = [col for col in X_train.columns if col not in numeric_features]

# 2. Define the transformation 'rules'
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

print("Preprocessing rules defined!")

Preprocessing rules defined!


In [9]:
from sklearn.linear_model import LogisticRegression

# Create the final "Model Pipeline"
# It says: First Preprocess, then run Logistic Regression
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# TRAIN THE MODEL (The "Learning" phase)
# It learns the patterns from the training set only
model_pipeline.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Ask the model to take the 'Final Exam'
y_pred = model_pipeline.predict(X_test)

# 2. See the results
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# 3. See the Weights (Coefficients)
# This shows exactly which features 'won'
feat_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
weights = model_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Feature': feat_names, 'Weight': weights}).sort_values(by='Weight', ascending=False)
print("\n--- Top Churn Drivers (Positive Weights) ---")
print(coef_df.head(5))
print("\n--- Top Retention Anchors (Negative Weights) ---")
print(coef_df.tail(5))

--- Classification Report ---
              precision    recall  f1-score   support

       False       0.84      0.88      0.86      1033
        True       0.63      0.55      0.59       374

    accuracy                           0.80      1407
   macro avg       0.74      0.72      0.73      1407
weighted avg       0.79      0.80      0.79      1407


--- Top Churn Drivers (Positive Weights) ---
                                 Feature    Weight
8      cat__internet_service_Fiber optic  0.944235
2                     num__total_charges  0.712817
6   cat__multiple_lines_No phone service  0.537359
26  cat__payment_method_Electronic check  0.381654
21             cat__streaming_movies_Yes  0.320984

--- Top Retention Anchors (Negative Weights) ---
                     Feature    Weight
17     cat__tech_support_Yes -0.392705
11  cat__online_security_Yes -0.402802
22    cat__contract_One year -0.667034
23    cat__contract_Two year -1.316150
0                num__tenure -1.476925
